# Lab 5 · Notebook 02 – Giả lập thiếu nhãn và supervised baseline

Chia **theo thời gian** thành FIT (60%), validation (20%), test (20%). **Chỉ ẩn nhãn trong FIT**; validation và test giữ nhãn thật, không dùng để tạo pseudo-label.

Supervised baseline chỉ học từ FIT còn nhãn. Dùng **Macro-F1** để quan sát hiệu quả hai lớp.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src" / "lab5.py").exists()), None)
assert ROOT, "Hãy chạy Jupyter trong repository FIT4112 Lab 5"
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
print("Repo:", ROOT)

In [ ]:
import pandas as pd
from src.lab5 import TARGET,FEATURES,chronological_split,hide_fit_labels,train_supervised,metrics
df=pd.read_csv(ROOT/"data/processed/prepared_demo_or_uci.csv",parse_dates=["datetime"])
fit,val,test=chronological_split(df)
MISSING_FRACTION=.75
masked=hide_fit_labels(fit,missing_fraction=MISSING_FRACTION)
print("FIT:",len(fit),"VAL:",len(val),"TEST:",len(test))
print("Train labels real:",masked[TARGET].notna().sum(),"unlabeled:",masked[TARGET].isna().sum())
assert masked[TARGET].isna().any() and val[TARGET].notna().all() and test[TARGET].notna().all()
baseline=train_supervised(masked)
print("Baseline validation:",metrics(baseline,val))

### Kiểm soát phân chia

- Tất cả bước lựa chọn thuật toán/ngưỡng **chỉ sử dụng validation**.
- Test được giữ nguyên cho đánh giá cuối. Không nhìn test trước khi chọn tau.
- Không cần thử nhiều tỷ lệ thiếu nhãn trong bài bắt buộc.

In [ ]:
assert fit.datetime.max()<val.datetime.min()<test.datetime.min()
assert not set(fit.index).intersection(set())  # no row transfer; chronological split checked above
save=ROOT/"data/processed"
# Không lưu fit nhãn thật dùng để mô phỏng: lưu bản đã ẩn
masked.to_csv(save/"masked_fit.csv",index=False)
val.to_csv(save/"validation.csv",index=False)
test.to_csv(save/"heldout_test.csv",index=False)
print("Saved training / validation / held-out test partitions")

### Câu hỏi
1. Có bao nhiêu phần trăm nhãn bị ẩn? Nhãn ẩn có được dùng để fit baseline không?
2. Vì sao không ẩn nhãn validation/test? Vì sao dùng Macro-F1 chứ không chỉ accuracy?
3. Có thể suy ra Self-Training chắc chắn tốt hơn supervised không?